In [1]:
#Load JailbreakBench behaviors dataset and convert to DataFrame
from datasets import load_dataset
import pandas as pd

print("Loading JailbreakBench behaviors...")
dataset = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors")

print(f"Dataset keys: {dataset.keys()}")
print(f"Dataset type: {type(dataset)}")

if hasattr(dataset, 'keys'):
    split_name = list(dataset.keys())[0]
    print(f"Using split: {split_name}")
    behaviors = dataset[split_name]
else:
    behaviors = dataset

behaviors_df = pd.DataFrame(behaviors)

print(f"\n✓ Loaded {len(behaviors_df)} behaviors")
print(f"Columns: {behaviors_df.columns.tolist()}")
print(f"\nFirst row:")
print(behaviors_df.iloc[0])

if 'goal' in behaviors_df.columns:
    print(f"\n✓ Has 'goal' column")
else:
    print(f"\nAvailable columns: {behaviors_df.columns.tolist()}")

In [2]:
#Set vLLM endpoint/model constants and test server connectivity
import requests
import json
import time
from datetime import datetime

VLLM_URL = "http://192.168.2.229:8000/v1/completions"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

try:
    test_response = requests.get("http://192.168.2.229:8000/health", timeout=5)
    if test_response.status_code == 200:
        print("✓ vLLM server is reachable, \n", test_response)
    else:
        print(f"✗ vLLM returned status {test_response.status_code}")
except Exception as e:
    print(f"✗ Cannot reach vLLM: {e}")
    print("Make sure vLLM is running on your WSL 2 machine at port 8000")

In [3]:
#Send each JailbreakBench behavior to vLLM and collect responses with latency
import requests
import pandas as pd
import time
from datetime import datetime

results = []
failed = []

start_idx = 0

print(f"Starting from behavior {start_idx}")
print(f"Total to process: {len(behaviors_df) - start_idx}\n")

for idx, row in behaviors_df.iloc[start_idx:].iterrows():
    goal = row['Goal'] if 'Goal' in row else row.get('Behavior', '')

    print("goal: ", goal)
    
    category = row.get('Category', 'Unknown')
    
    payload = {
        "model": MODEL_NAME,
        "prompt": goal,
        "max_tokens": 100,
        "temperature": 0.7
    }
    
    try:
        start_time = time.time()
        response = requests.post(VLLM_URL, json=payload, timeout=60)
        latency = time.time() - start_time
        
        data = response.json()

        print("response: \n", data)
        
        results.append({
            'Idx': idx,
            'Timestamp': datetime.now().isoformat(),
            'Goal': goal[:100],
            'Category': category,
            'Response': data['choices'][0]['text'][:200],
            'Response_Tokens': data['usage']['completion_tokens'],
            'Latency_Seconds': round(latency, 2)
        })
        
        if (idx + 1) % 10 == 0:
            print(f"✓ Completed {idx + 1}/{len(behaviors_df)} - Last latency: {latency:.2f}s")
        
        time.sleep(0.5)
        
    except Exception as e:
        failed.append({'Idx': idx, 'Goal': goal[:50], 'Error': str(e)})
        print(f"✗ Error on {idx}: {e}")

print(f"\n✓ Completed {len(results)} queries")
print(f"✗ Failed {len(failed)} queries")

results_df = pd.DataFrame(results)
print(f"\nResults summary:")
print(results_df.groupby('Category').size())

In [ ]:
#Save results to JSONL and print summary stats by category
results_df.to_json('jailbreak_results.jsonl', orient='records', lines=True)
print(f"✓ Saved to jailbreak_results.jsonl")

print(f"\n=== STATISTICS ===")
print(f"Total queries: {len(results_df)}")
print(f"Average latency: {results_df['Latency_Seconds'].mean():.2f}s")
print(f"Avg tokens generated: {results_df['Response_Tokens'].mean():.0f}")

print(f"\nBy category:")
print(results_df.groupby('Category').agg({
    'Latency_Seconds': 'mean',
    'Response_Tokens': 'mean'
}).round(2))

if len(failed) > 0:
    print(f"\n✗ Failed queries:")
    for f in failed[:5]:
        print(f"  {f['idx']}: {f['error'][:50]}")

In [ ]:
#Load jailbreak results JSONL and print latency/token stats grouped by category
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("Loading datasets...")

with open('Logs/jailbreak_results.jsonl', 'r') as f:
    jailbreak_results = [json.loads(line) for line in f]

jailbreak_df = pd.DataFrame(jailbreak_results)
jailbreak_df['Timestamp'] = pd.to_datetime(jailbreak_df['Timestamp'])

print(f"✓ Loaded {len(jailbreak_df)} JailbreakBench results")
print(f"  Time range: {jailbreak_df['Timestamp'].min()} to {jailbreak_df['Timestamp'].max()}")

print(f"\n=== ANALYSIS BY CATEGORY ===\n")

for category in sorted(jailbreak_df['Category'].unique()):
    subset = jailbreak_df[jailbreak_df['Category'] == category]
    print(f"{category.upper()}")
    print(f"  Count: {len(subset)}")
    print(f"  Avg Latency: {subset['Latency_Seconds'].mean():.2f}s")
    print(f"  Avg Tokens: {subset['Response_Tokens'].mean():.0f}")
    print(f"  Token efficiency: {(subset['Response_Tokens'] / subset['Latency_Seconds']).mean():.1f} tokens/sec")
    print()

jailbreak_df.to_json('logs/jailbreak_analysis.jsonl', orient='records', lines=True)
print("✓ Saved analysis to logs/jailbreak_analysis.jsonl")

In [ ]:
#Display full jailbreak results DataFrame
display(jailbreak_df)

In [ ]:
#Install required packages
!pip install datasets huggingface_hub pandas matplotlib

In [ ]:
#Install ipywidgets and enable notebook extension
!pip install ipywidgets
!jupyter nbextension enable --py widgetsnbextension

In [ ]:
#Log in to HuggingFace Hub
from huggingface_hub import login
login()

In [ ]:
#Load lmsys-chat-1m, extract first user turn, filter to clean English prompts, sample 1000 benign
from datasets import load_dataset
import pandas as pd
from huggingface_hub import login
login()

dataset = load_dataset("lmsys/lmsys-chat-1m", split="train")

def extract_first_user_turn(example):
    #Return prompt and language from the first user message in a conversation
    for msg in example["conversation"]:
        if msg["role"] == "user":
            return {"prompt": msg["content"], "language": example.get("language", "unknown")}
    return {"prompt": None, "language": None}

dataset = dataset.map(extract_first_user_turn)

df = dataset.to_pandas()
df = df[
    (df["language"] == "English") &
    (df["openai_moderation"].apply(lambda x: not x["flagged"] if isinstance(x, dict) else True)) &
    (df["prompt"].str.len() > 30) &
    (df["prompt"].str.len() < 2000) &
    (df["prompt"].notna())
]

benign_df = df.sample(n=1000, random_state=42)[["prompt"]].reset_index(drop=True)
benign_df["label"] = 0

print(f"Benign samples: {len(benign_df)}")
print(f"Avg prompt length: {benign_df['prompt'].str.len().mean():.0f} chars")

In [ ]:
#Print shape, label counts, null/duplicate checks, and plot prompt length histogram
import matplotlib.pyplot as plt

print(f"Shape: {benign_df.shape}")
print(f"Label distribution:\n{benign_df['label'].value_counts()}")
print(f"\nNull prompts: {benign_df['prompt'].isna().sum()}")
print(f"Duplicate prompts: {benign_df['prompt'].duplicated().sum()}")

benign_df["prompt_len"] = benign_df["prompt"].str.len()
print(f"\nPrompt length stats:")
print(benign_df["prompt_len"].describe())

benign_df["prompt_len"].hist(bins=50, figsize=(10,4))
plt.title("Benign Prompt Length Distribution")
plt.xlabel("Character length")
plt.ylabel("Count")
plt.show()

benign_df.head(5)

In [ ]:
#Save benign prompts DataFrame to CSV
benign_df.to_csv("benign_prompts.csv", index=False)
print("Saved.")

In [ ]:
#Send each benign prompt to vLLM endpoint one at a time with a delay
import requests
import time
import json

ENDPOINT = "http://192.168.2.229:8001/v1/completions"

def send_prompt(prompt):
    #Post a prompt to vLLM and return the JSON response
    payload = {
        "prompt": prompt,
        "max_tokens": 256,
        "temperature": 0.0
    }
    try:
        response = requests.post(ENDPOINT, json=payload, timeout=60)
        return response.json()
    except Exception as e:
        print(f"Error: {e}")
        return None

for i, row in benign_df.iterrows():
    print(f"[{i+1}/1000] Sending prompt...")
    result = send_prompt(row["prompt"])
    time.sleep(0.5)

print("Done.")

In [ ]:
#Send a single test prompt and print the raw response
test = send_prompt(benign_df.iloc[0]["prompt"])
print(json.dumps(test, indent=2))

In [ ]:
#Check which models are available on the vLLM server
print(requests.get("http://192.168.2.229:8001/v1/models").json())